# Stromal sub-clustering (3-layer pipeline)

This notebook continues from `04_cell_type_annotation.ipynb`: it subsets the `Stromal` compartment (from `lv1_annotation`), re-clusters it at finer resolution, and annotates the resulting sub-clusters using the same three complementary layers of evidence.

1. **Canonical/known marker annotation** — score each sub-cluster against the stromal/vessel-relevant curated marker sets.
2. **DEG-driven manual annotation** — pull top DEGs per sub-cluster, cross-check against literature/Protein Atlas, and record a manual label.
3. **Functional enrichment (GO/KEGG/Reactome)** — run over-representation analysis on sub-cluster DEGs to support/refine the label.

Outputs are written to `results/xenium_5k/annotation/subclustering/stromal/`.

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import spatialdata as sd
import spatialdata_plot
import gseapy as gp

%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

GROUP = 'Stromal'
annot_dir = '../../results/xenium_5k/annotation/subclustering/stromal'
os.makedirs(annot_dir, exist_ok=True)

In [ ]:
# Load the lv1-annotated object saved at the end of notebook 04 and subset to this compartment
sdata = sd.read_zarr('../../data/spatialdata/xenium_prime_5k_lv1_annotated.zarr')
adata_full = sdata['table']
adata = adata_full[adata_full.obs['lv1_annotation'] == GROUP].copy()
print(f'{adata.n_obs} cells in the {GROUP} compartment (of {adata_full.n_obs} total)')

## Sub-clustering

Fresh HVG -> PCA -> neighbors -> Leiden on just this compartment's cells (mirrors `02_Xenium_5k_downstream_qc_filter_clustering_python.ipynb`'s parameters). Bump `sub_resolution` if sub-types still look merged.

In [ ]:
sub_resolution = 0.5

sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=2000)
sc.pp.pca(adata, n_comps=30)
sc.pp.neighbors(adata, metric='cosine')
sc.tl.leiden(adata, key_added='sub_leiden', flavor='igraph', n_iterations=-1, resolution=sub_resolution)
sc.tl.umap(adata)

adata.obs['sub_leiden'] = adata.obs['sub_leiden'].astype('category')
sub_clusters = adata.obs['sub_leiden'].cat.categories.tolist()
print(f'{len(sub_clusters)} sub-clusters, {adata.n_obs} cells')

In [ ]:
sc.pl.umap(adata, color='sub_leiden', legend_loc='on data', legend_fontsize=8, legend_fontoutline=2)

## Layer 1 — Canonical/known marker annotation

Reuses the curated marker dict from `04_cell_type_annotation.ipynb`, filtered down to the signatures relevant to this compartment (plus `Proliferating`, useful in every compartment).

In [ ]:
canonical_markers_all = {
    'Epithelial_AT1': ['AGER', 'PDPN', 'CAV1'],
    'Epithelial_AT2': ['LAMP3', 'ABCA3', 'SLC34A2', 'TCIM', 'SFTPC', 'SFTPB', 'SFTPA1', 'NAPSA'],
    'Epithelial_Airway_Ciliated': ['FOXJ1', 'TPPP3', 'PIFO'],
    'Epithelial_Airway_Club_Secretory': ['SCGB1A1', 'SCGB3A1', 'MUC5B'],
    'Epithelial_Tumor_Malignant': ['EPCAM', 'KRT8', 'KRT18', 'KRT19'],
    'Epithelial_LUAD (Adenocarcinoma)': ['NAPSA', 'NKX2-1', 'SFTPC', 'SFTPB', 'MUC5AC', 'MUC5B', 'KRT7'],
    'Lymphoid_T_cell_CD4+': ['CD3D', 'CD3E', 'CD2', 'TRAC', 'CD4'],
    'Lymphoid_T_cell_CD8+': ['CD3D', 'CD3E', 'CD2', 'TRAC', 'CD8A', 'CD8B'],
    'Lymphoid_NK_cell': ['NKG7', 'GNLY', 'KLRD1'],
    'Lymphoid_B_cell': ['MS4A1', 'CD79A', 'CD79B'],
    'Lymphoid_Plasma_cell': ['MZB1', 'JCHAIN', 'IGHG1'],
    'Myeloid_Macrophage_M1': ['CD68', 'CD86', 'CD80', 'IL1B', 'TNF', 'NOS2', 'IL6', 'IL12A'],
    'Myeloid_Macrophage_M2': ['CD68', 'CD163', 'MRC1', 'CCL18', 'MSR1', 'IL10', 'TGM2', 'TGFB1'],
    'Myeloid_Monocyte': ['CD14', 'CD74', 'LYZ', 'ITGAM'],
    'Myeloid_Neutrophil': ['FCGR3B', 'FMLP2', 'CEBPE', 'MPO', 'ELANE', 'AZU1', 'LILRA5', 'MXD1', 'CSF3R'],
    'Myeloid_Dendritic_cell': ['CD74', 'FCER1A', 'CST3', 'AIF1'],
    'Myeloid_Mast_cell': ['TPSAB1', 'TPSB2', 'CPA3', 'CMA1', 'KIT'],
    'Stromal_Fibroblast': ['COL1A1', 'COL1A2', 'DCN', 'PDGFRA'],
    'Stromal_Smooth_muscle/Pericyte': ['ACTA2', 'MYH11', 'PDGFRB'],
    'Vessels_EC': ['PECAM1', 'VWF', 'CLDN5'],
    'Proliferating': ['MKI67', 'TOP2A', 'PCNA', 'CDK1', 'UBE2C', 'BIRC5', 'CCNB1', 'CCNB2', 'MCM2', 'MCM5'],
}

# Keep only the marker sets relevant to this compartment (+ Proliferating)
group_prefixes = ('Stromal_',)
extra_keys = {'Vessels_EC', 'Proliferating'}
canonical_markers = {
    k: v for k, v in canonical_markers_all.items()
    if k.startswith(group_prefixes) or k in extra_keys
}

# Keep only genes present in the panel, and drop cell types left with no genes
canonical_markers = {
    cell_type: [g for g in genes if g in adata.var_names]
    for cell_type, genes in canonical_markers.items()
}
missing = {k: v for k, v in canonical_markers.items() if len(v) == 0}
if missing:
    print('Cell types with no marker genes found in panel (dropped):', list(missing.keys()))
canonical_markers = {k: v for k, v in canonical_markers.items() if len(v) > 0}
canonical_markers

In [ ]:
adata.obs.drop(columns=[c for c in adata.obs.columns if c.startswith('score_')], inplace=True, errors='ignore')
for cell_type, genes in canonical_markers.items():
    sc.tl.score_genes(adata, gene_list=genes, score_name=f'score_{cell_type}', layer='lognorm')

score_cols = [f'score_{ct}' for ct in canonical_markers]
cluster_scores = adata.obs.groupby('sub_leiden', observed=True)[score_cols].mean()
cluster_scores.columns = list(canonical_markers.keys())
cluster_scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.4 * len(cluster_scores) + 2))
im = ax.imshow(cluster_scores.values, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(cluster_scores.columns)))
ax.set_xticklabels(cluster_scores.columns, rotation=90)
ax.set_yticks(range(len(cluster_scores)))
ax.set_yticklabels(cluster_scores.index)
ax.set_xlabel('Canonical cell type signature')
ax.set_ylabel('sub_leiden cluster')
fig.colorbar(im, ax=ax, label='mean score_genes')
plt.tight_layout()

In [ ]:
layer1_annotation = pd.DataFrame({
    'cluster': cluster_scores.index,
    'canonical_label': cluster_scores.idxmax(axis=1).values,
    'canonical_score': cluster_scores.max(axis=1).values,
}).reset_index(drop=True)
layer1_annotation

In [ ]:
def plot_subcluster_score(adata, sdata, cell_type_name, figsize=(12.8, 9.6)):
    """Plot a sub-cluster marker score spatially by projecting it onto the full sdata table (in-memory only)."""
    score_col = f'score_{cell_type_name}'
    if score_col not in adata.obs.columns:
        print(f"Score column '{score_col}' not found in adata.obs")
        print('Available score columns:', [c for c in adata.obs.columns if c.startswith('score_')])
        return

    plot_col = re.sub(r'[^A-Za-z0-9_.-]', '_', score_col)
    full_table = sdata.tables['table']
    full_table.obs[plot_col] = adata.obs[score_col].reindex(full_table.obs.index)

    sc.pl.umap(adata, color=score_col, cmap='viridis')
    sdata.pl.render_shapes('cell_boundaries', color=plot_col, cmap='viridis').pl.show(figsize=figsize)

# Example usage:
# plot_subcluster_score(adata, sdata, 'Stromal_Fibroblast')

In [ ]:
plot_subcluster_score(adata, sdata, 'Proliferating')

## Layer 2 — DEG-driven manual/literature annotation

Compute DEGs per sub-cluster, pull the top 5–10 genes, then cross-check them against literature / [Human Protein Atlas](https://www.proteinatlas.org) to assign a label.

Manual calls are stored in an editable CSV (`layer2_manual_annotation.csv`) so re-running this notebook doesn't wipe out annotation work — edit the CSV directly and re-run the load cell to pick up changes.

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='sub_leiden', layer='lognorm', pts=True)

In [ ]:
n_top_degs = 10
deg_summary = pd.concat(
    [sc.get.rank_genes_groups_df(adata, group=c).head(n_top_degs).assign(cluster=c) for c in sub_clusters],
    ignore_index=True,
)
deg_summary = deg_summary[['cluster', 'names', 'scores', 'logfoldchanges', 'pvals_adj', 'pct_nz_group', 'pct_nz_reference']]
deg_summary.to_csv(f'{annot_dir}/deg_summary.csv', index=False)
deg_summary.head(10)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5, swap_axes=True, figsize=(8, max(6, 0.6 * len(sub_clusters))))

In [ ]:
layer2_path = f'{annot_dir}/layer2_manual_annotation.csv'
top_degs_per_cluster = (
    deg_summary.groupby('cluster', observed=True)['names']
    .apply(lambda s: ', '.join(s))
    .reindex(sub_clusters)
)

if os.path.exists(layer2_path):
    layer2_manual_annotation = pd.read_csv(layer2_path, dtype=str).set_index('cluster')
    layer2_manual_annotation['top_degs'] = top_degs_per_cluster
    layer2_manual_annotation = layer2_manual_annotation.reset_index()
else:
    layer2_manual_annotation = pd.DataFrame({
        'cluster': sub_clusters,
        'top_degs': top_degs_per_cluster.values,
        'literature_notes': '',
        'layer2_label': '',
    })

layer2_manual_annotation.to_csv(layer2_path, index=False)
print(f'Edit {layer2_path} to fill in literature_notes / layer2_label, then re-run this cell to reload.')
layer2_manual_annotation

#### Cross-check top DEGs against HPA marker genes

Fisher's exact test (via `celltypegenomics`) of each sub-cluster's top DEGs against Human Protein Atlas cell-type marker gene sets, as an additional cross-check for the manual calls above.

In [ ]:
from celltypegenomics import celltypefishertest

symbol_to_ensembl = adata.var['gene_ids'].to_dict()


def hpa_fisher_test(cluster, n=n_top_degs):
    genes = deg_summary.loc[deg_summary['cluster'] == cluster, 'names'].head(n)
    ensembl_ids = [symbol_to_ensembl[g] for g in genes if g in symbol_to_ensembl]
    result = celltypefishertest(ensembl_ids, hpa_marker_genes=True)
    return result.sort_values('adjusted_p_value').assign(cluster=cluster)


hpa_annotation = pd.concat([hpa_fisher_test(c) for c in sub_clusters], ignore_index=False)
hpa_annotation.to_csv(f'{annot_dir}/layer2_hpa_fishertest.csv')
hpa_annotation.groupby('cluster').head(3)

In [ ]:
hpa_best_per_cluster = (
    hpa_annotation.reset_index(names='cell_type')
    .sort_values('adjusted_p_value')
    .groupby('cluster', sort=False)
    .first()
    .reindex(sub_clusters)[['cell_type', 'adjusted_p_value', 'odds_ratio']]
)
hpa_best_per_cluster

## Layer 3 — Functional enrichment (GO / KEGG / Reactome)

Over-representation analysis (Enrichr) on the top DEGs of each sub-cluster, to support/refine the cell type call with pathway-level evidence.

In [ ]:
gene_sets = ['GO_Biological_Process_2023', 'KEGG_2021_Human', 'Reactome_2022']
n_genes_enrichment = 50


def top_deg_genes(cluster, n=n_genes_enrichment, padj_thresh=0.05):
    df = sc.get.rank_genes_groups_df(adata, group=cluster)
    df = df[(df['pvals_adj'] < padj_thresh) & (df['logfoldchanges'] > 0)]
    return df.sort_values('scores', ascending=False)['names'].head(n).tolist()


def run_enrichment(gene_list, cluster, gene_sets=gene_sets):
    if len(gene_list) == 0:
        return pd.DataFrame()
    try:
        enr = gp.enrich_multi(gene_list=gene_list, gene_sets=gene_sets, organism='human', outdir=None)
    except AttributeError:
        # older/newer gseapy versions expose enrichr per-library instead of enrich_multi
        enr = pd.concat(
            [gp.enrichr(gene_list=gene_list, gene_sets=[gs], organism='human', outdir=None).results.assign(Gene_set=gs)
             for gs in gene_sets],
            ignore_index=True,
        )
    enr = enr.results if hasattr(enr, 'results') else enr
    enr['cluster'] = cluster
    return enr

In [ ]:
%%time
enrichment_results = {}
for c in sub_clusters:
    genes = top_deg_genes(c)
    res = run_enrichment(genes, c)
    if not res.empty:
        enrichment_results[c] = res

layer3_enrichment = pd.concat(enrichment_results.values(), ignore_index=True) if enrichment_results else pd.DataFrame()
layer3_enrichment.to_csv(f'{annot_dir}/layer3_functional_enrichment.csv', index=False)
layer3_enrichment.head(10)

In [ ]:
# Barplot of the top enriched terms for a single sub-cluster (edit cluster id as needed)
cluster_to_plot = sub_clusters[0]
top_terms = (
    layer3_enrichment[layer3_enrichment['cluster'] == cluster_to_plot]
    .sort_values('Adjusted P-value')
    .head(10)
)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_terms['Term'], -np.log10(top_terms['Adjusted P-value']))
ax.invert_yaxis()
ax.set_xlabel('-log10(adjusted p-value)')
ax.set_title(f'Top enriched terms — sub-cluster {cluster_to_plot}')
plt.tight_layout()

In [ ]:
top_enriched_terms_per_cluster = (
    layer3_enrichment.sort_values('Adjusted P-value')
    .groupby('cluster', observed=True)['Term']
    .apply(lambda s: '; '.join(s.head(3)))
    .reindex(sub_clusters)
)

## Consolidation

Merge all three layers into one summary table per sub-cluster.

In [ ]:
n_cells_per_cluster = adata.obs['sub_leiden'].value_counts().reindex(sub_clusters)

subcluster_annotation_summary = (
    layer1_annotation.set_index('cluster')
    .join(layer2_manual_annotation.set_index('cluster'))
    .assign(
        n_cells=n_cells_per_cluster,
        top_enriched_terms=top_enriched_terms_per_cluster,
    )
    .reset_index()
)
subcluster_annotation_summary.to_csv(f'{annot_dir}/subcluster_annotation_summary.csv', index=False)
subcluster_annotation_summary

Fill in `subcluster_labels` below once the layer 1-3 evidence above has been reviewed, then re-run the remaining cells to plot the final sub-type calls (in-memory only — this notebook does not persist a new zarr store).

In [ ]:
# TODO: fill in final sub-type labels per sub_leiden cluster, informed by layers 1-3 above
subcluster_labels = {
    # 'Stromal_Fibroblast': ['0'],
}

cluster_to_label = {c: label for label, cs in subcluster_labels.items() for c in cs}
adata.obs['sub_cell_type'] = adata.obs['sub_leiden'].map(cluster_to_label).astype('category')

sc.pl.umap(adata, color='sub_cell_type', legend_loc='on data', legend_fontsize=8, legend_fontoutline=2)

In [ ]:
plot_col = 'sub_cell_type'
full_table = sdata.tables['table']
full_table.obs[plot_col] = adata.obs[plot_col].reindex(full_table.obs.index)

sdata.tables['table'].obs['region'] = 'cell_boundaries'
sdata.set_table_annotates_spatialelement('table', region='cell_boundaries')
sdata.pl.render_shapes('cell_boundaries', color=plot_col).pl.show(figsize=(12.8, 9.6))